<a href="https://colab.research.google.com/github/gautamkr1876/AIML_ClassNotes/blob/main/8.%20Agentic%20AI%20Systems/3.Docker%20%26%20Containerization%20for%20AI%20Applications/L3_%20Containerization_with_Docker_for_AI_Engineering.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Containerization with Docker for AI Engineering

---

### What You'll Learn Today

| # | Topic |
|---|-------|
| 1 | Why Containerization? (The "Works on My Machine" Problem) |
| 2 | Docker Refresher: Images, Containers, Registries |
| 3 | Build a Small RAG App From Scratch (our test subject for today) |
| 4 | Dockerfile Basics and Best Practices for Python AI Apps |
| 5 | Multi-Stage Builds: Shrinking a 2GB Image to ~500MB |
| 6 | Docker Compose for Multi-Service Applications (App + Vector DB + Monitoring) |
| 7 | Environment Variables and Secrets Management |
| 8 | Packaging and Sharing RAG Applications |

**Live Hands-On Exercise (in class):** Containerize the RAG application we build today, personalize it, push the image to Docker Hub, and have a peer pull and run it on their machine.

**Everything in this notebook is standalone.** We write every file we need (sample documents, ingestion script, API server, Dockerfiles, compose file) inside this class. You need nothing from previous sessions except Docker installed and an OpenAI API key.

---

### Session Plan (2.5 hours)

| Time | Block |
|---|---|
| 0:00 - 0:15 | Why Containerization + Docker Refresher |
| 0:15 - 0:35 | Build the RAG App We Are Going to Ship |
| 0:35 - 1:00 | Dockerfile Basics and Best Practices for Python AI Apps |
| 1:00 - 1:20 | Multi-Stage Builds |
| 1:20 - 1:25 | Break |
| 1:25 - 1:50 | Docker Compose: App + Vector DB + Monitoring |
| 1:50 - 2:05 | Environment Variables and Secrets |
| 2:05 - 2:15 | Packaging and Sharing RAG Applications |
| 2:15 - 2:30 | Live Exercise: Share Your RAG Image with a Peer |

> **Instructor's note:** The live exercise needs a free Docker Hub account per student and an OpenAI API key per student. Announce both at the start of class so accounts get created during the break, not during the exercise.

## The Problem We Are Solving Today

Picture this. You spend a week building a RAG application: document ingestion, chunking, embeddings, a vector store, retrieval, an LLM call, all wired behind a FastAPI endpoint. It runs beautifully on your laptop.

Now zip up the project folder and send it to the person sitting next to you. Will it run on their machine?

Almost certainly not. Here is the usual sequence of events:

```
ModuleNotFoundError: No module named 'chromadb'
```

They install it. Then:

```
RuntimeError: Your system has an unsupported version of sqlite3.
Chroma requires sqlite3 >= 3.35.0
```

They patch that. Then `sentence-transformers` pulls a different `torch` build on their machine, inference is suddenly 10x slower, and on Windows the file paths in the ingestion script break entirely.

You shipped your **code**, but not your **environment**. A `requirements.txt` pins Python packages, but it says nothing about the Python version, the OS, system libraries like sqlite3 or libgomp, or how supporting services (like a vector DB server) should be started.

| What it captures | requirements.txt + venv | Docker |
|---|---|---|
| Python dependencies | Yes | Yes |
| Python version | No (hopefully documented) | Yes |
| System / OS libraries (sqlite3, libgomp, etc.) | No | Yes |
| OS-level behavior | No | Yes |
| Supporting services (vector DB, monitoring) | No | Yes (with Compose) |
| Runs identically everywhere | No | Yes |

> **Mental model:** requirements.txt describes your Python packages. Docker describes the entire machine your app runs on.

**Today, Docker closes that gap.** By the end of this class, "send me your project" becomes "pull my image", and it just runs. We will prove it live: in the last 15 minutes, a peer runs your image on their machine with zero setup conversation.

---
## 1. Why Containerization?

### Why AI Apps Suffer From This More Than Most

Every software team hits environment drift. AI applications get hit harder, for very specific reasons:

- **Heavy native dependencies.** `torch`, `tokenizers`, `onnxruntime`, `faiss` all ship compiled binaries. The wheel that installs on your Mac is not the wheel that installs on a Linux server.
- **System library pinning.** ChromaDB needs sqlite3 >= 3.35. Many LTS Linux distributions ship older versions. Your code is correct; the OS is not.
- **Version sensitivity.** A minor bump in an embedding library can silently change vector values. Retrieval quality degrades and nobody knows why.
- **Multiple moving parts.** A RAG app is rarely one process. There is the API, the vector DB, maybe a monitoring stack. "Install my app" really means "reproduce my whole setup".

### What Docker Actually Is

**Docker** is a platform for building, shipping, and running applications inside **containers**: lightweight, isolated, self-contained execution environments.

A Docker container for a RAG app bundles:

- The application code (server, ingestion script)
- The exact Python runtime (3.11.x, not "whatever is installed")
- All Python dependencies, pinned
- All OS-level libraries (a sqlite3 that ChromaDB is happy with)
- The startup command

into a single artifact, the **image**, that runs identically on:

- Your laptop
- A teammate's machine (the live exercise today proves this)
- A CI/CD runner
- A cloud VM or Kubernetes cluster

### Docker vs Virtual Machines (Quick Comparison)

Both provide isolation, but the mechanism is different. A VM emulates an entire machine including a guest OS kernel (gigabytes, minutes to boot). Containers share the host OS kernel and isolate only the user space (megabytes of overhead, sub-second startup).

```
VIRTUAL MACHINE                      DOCKER CONTAINER
+---------------------------+       +--------+ +--------+ +--------+
|       Application         |       | RAG    | | Chroma | | Prom-  |
+---------------------------+       | API    | | DB     | | etheus |
|   Guest OS (full kernel)  |       +--------+-+--------+-+--------+
+---------------------------+       |        Docker Engine         |
+---------------------------+       +------------------------------+
|       Hypervisor          |       |   Host OS Kernel (shared)    |
+---------------------------+       +------------------------------+
|        Host OS            |       |     Physical Hardware        |
+---------------------------+       +------------------------------+
|     Physical Hardware     |
+---------------------------+
```

| | Docker Container | Virtual Machine |
|---|---|---|
| Size | MBs to a few GB | Often 10s of GB |
| Startup | Sub-second | Minutes |
| Kernel | Shares host kernel | Runs its own |
| Typical use here | Packaging the RAG API | Running a different OS entirely |

For shipping AI applications, containers are the industry default. Every major LLM serving stack (vLLM, TGI, Ollama server deployments) ships as a Docker image.

### Install Docker (If You Have Not Already)

The easiest path is **Docker Desktop**, which installs the daemon, the CLI, and a GUI.

**Download:** https://www.docker.com/products/docker-desktop

| OS | Instructions |
|---|---|
| **Mac** | Download the `.dmg`, drag to Applications, launch |
| **Windows** | Requires the WSL2 backend. In PowerShell as Administrator run `wsl --install`, restart, then install Docker Desktop |
| **Linux** | Install Docker Engine directly. To skip `sudo` on every command: `sudo usermod -aG docker $USER` then `newgrp docker` |

### Verify Your Installation

In [ ]:
# Run these in your terminal (not in this notebook).
# Shown here for reference.

print("""
# 1. Check Docker version
$ docker --version
Docker version 29.2.1, build a5c7197

# 2. The canonical smoke test
$ docker run hello-world
Hello from Docker!
This message shows that your installation appears to be working correctly.
""")

---
## 2. Docker Refresher: Images, Containers, Registries

A fast recap so the vocabulary is fresh. If this section is all new to you, flag it now and review the fundamentals notebook after class.

### The Core Analogy

| Concept | Analogy (OOP) | Docker |
|---|---|---|
| Blueprint | Class definition | **Dockerfile** |
| Built artifact | Compiled class | **Docker Image** (read-only, layered) |
| Running instance | Object | **Docker Container** (image + writable layer) |

One image, many containers. That is how you scale a service: same image on ports 8000, 8001, 8002.

### Registries

A **registry** is GitHub for images. **Docker Hub** (hub.docker.com) is the default public one; companies use private registries (AWS ECR, GitHub Container Registry) for production. Today's exercise uses Docker Hub because it is free and your peer can pull from it instantly.

Image naming convention:

```
username/repository:tag

python:3.11-slim                <- official image (no username prefix)
chromadb/chroma:0.5.23          <- vendor image
yourusername/rag-app:1.0.0      <- what you will push today
```

### The Commands You Will Use Constantly

In [ ]:
# Core Docker CLI reference. Run these in your terminal.

docker_commands = {
    "IMAGES": [
        ("docker images",                       "List local images"),
        ("docker pull python:3.11-slim",        "Download an image from Docker Hub"),
        ("docker rmi rag-app:1.0.0",            "Remove a local image"),
        ("docker history rag-app:1.0.0",        "Show the size of every layer (we use this today)"),
    ],
    "CONTAINERS": [
        ("docker run -d -p 8000:8000 --name rag rag-app:1.0.0", "Start in background, map port, name it"),
        ("docker ps",                           "List running containers"),
        ("docker ps -a",                        "List all containers, including stopped"),
        ("docker logs -f rag",                  "Stream a container's logs"),
        ("docker exec -it rag bash",            "Open a shell inside a running container"),
        ("docker stop rag && docker rm rag",    "Stop and remove"),
        ("docker rm -f rag",                    "Force-stop and remove in one go"),
    ],
    "REGISTRY": [
        ("docker login",                        "Authenticate with Docker Hub"),
        ("docker tag rag-app:1.0.0 user/rag-app:1.0.0", "Tag for pushing"),
        ("docker push user/rag-app:1.0.0",      "Push to Docker Hub"),
        ("docker pull user/rag-app:1.0.0",      "Pull from Docker Hub"),
    ],
    "CLEANUP": [
        ("docker system prune",                 "Remove stopped containers, dangling images, unused networks"),
        ("docker system df",                    "See how much disk Docker is using (AI images add up fast)"),
    ],
}

for category, commands in docker_commands.items():
    print()
    print("=" * 70)
    print(f"  {category}")
    print("=" * 70)
    for cmd, desc in commands:
        print(f"  $ {cmd}")
        print(f"      {desc}")

### Checkpoint 1

Before moving on, make sure you can answer these:

1. What is the difference between an image and a container?
2. If `docker ps` shows nothing but `docker ps -a` shows three containers, what does that mean?
3. Why is `python:3.11-slim` preferred over `python:3.11` for our apps? (Hint: roughly 900MB vs 130MB base.)
4. Why should you avoid `python:3.11-alpine` for AI workloads even though it is smaller? (Hint: musl libc breaks many compiled ML wheels.)

---
## 3. Build a Small RAG App From Scratch

We need a real application to containerize, so we build one right now. It is deliberately small but structurally identical to a production RAG service. The complete project:

```
rag-app/
|-- docs/                  <- the knowledge base (3 small text files we write below)
|   |-- 01_containers.txt
|   |-- 02_images_layers.txt
|   |-- 03_rag_basics.txt
|-- ingest.py              <- chunks the docs, embeds them, loads them into ChromaDB
|-- app.py                 <- FastAPI server: /health and /ask
|-- requirements.txt
|-- .dockerignore
|-- Dockerfile             <- written in Section 4
```

The knowledge base is self-referential on purpose: the documents are about Docker and RAG, so once it runs you can ask the app questions about today's class and it will answer from its own index. Run the next five cells in order; each one writes a file to disk.

In [ ]:
# Cell 1 of 5: write the knowledge base documents

import os
os.makedirs("docs", exist_ok=True)

doc_containers = """What is a container?

A container is a lightweight, isolated execution environment that packages an
application together with its dependencies, runtime, and system libraries.
Unlike a virtual machine, a container shares the host operating system kernel,
which makes it start in under a second and keeps its overhead small.

Why containers matter for AI applications.

AI applications depend on compiled libraries such as torch and tokenizers,
on specific system libraries such as sqlite3 for ChromaDB, and often on
several cooperating services. Containers capture all of this in one shippable
artifact, which removes the works-on-my-machine problem.
"""

doc_images = """Images and layers.

A Docker image is a read-only, layered filesystem snapshot built from a
Dockerfile. Every instruction in the Dockerfile creates a layer. Layers are
cached: if an instruction and its inputs have not changed, Docker reuses the
cached layer instead of rebuilding it.

The most important caching rule.

Copy your requirements file and install dependencies before copying your
application code. Code changes daily, dependencies change weekly. With this
ordering, a code-only change rebuilds in seconds because the expensive
pip install layer is served from cache.

Multi-stage builds.

A multi-stage build uses one stage to compile and install dependencies and a
second clean stage that copies in only the finished result. Build tools and
caches never reach the final image, which is how a 2GB Python AI image
shrinks to roughly 500MB.
"""

doc_rag = """What is Retrieval-Augmented Generation?

RAG combines a retrieval step with a generation step. Documents are split
into chunks, each chunk is converted into an embedding vector, and the
vectors are stored in a vector database such as ChromaDB. At question time,
the question is embedded, the most similar chunks are retrieved, and a large
language model writes an answer grounded in those chunks.

Why RAG instead of fine-tuning?

RAG lets you update knowledge by re-ingesting documents instead of retraining
a model. It also makes answers auditable, because every answer can cite the
chunks it was built from.
"""

for name, content in [
    ("01_containers.txt", doc_containers),
    ("02_images_layers.txt", doc_images),
    ("03_rag_basics.txt", doc_rag),
]:
    with open(os.path.join("docs", name), "w") as f:
        f.write(content)
    print(f"docs/{name} written ({len(content)} chars)")

In [ ]:
# Cell 2 of 5: write the ingestion script
# Note the get_client() function. If CHROMA_HOST is set, we talk to a Chroma
# SERVER (Compose mode, Section 6). If it is not set, we use Chroma in
# EMBEDDED mode, writing the index to a local folder. One script, two modes,
# controlled purely by the environment. Remember this pattern.

ingest_script = '''# ingest.py - chunk docs/, embed, load into ChromaDB
import glob
import os

import chromadb
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = os.getenv("COLLECTION_NAME", "demo_docs")
EMBED_MODEL = os.getenv("EMBED_MODEL", "all-MiniLM-L6-v2")
DOCS_DIR = os.getenv("DOCS_DIR", "docs")


def get_client():
    host = os.getenv("CHROMA_HOST")
    if host:
        # server mode: a separate Chroma container or machine
        return chromadb.HttpClient(host=host, port=int(os.getenv("CHROMA_PORT", "8000")))
    # embedded mode: index lives in a local folder inside this environment
    return chromadb.PersistentClient(path=os.getenv("CHROMA_PATH", "chroma_db"))


def chunk_text(text, max_chars=600):
    # greedy paragraph packing: simple, good enough for a demo corpus
    paragraphs = [p.strip() for p in text.split("\\n\\n") if p.strip()]
    chunks, current = [], ""
    for para in paragraphs:
        if current and len(current) + len(para) > max_chars:
            chunks.append(current.strip())
            current = ""
        current += para + "\\n\\n"
    if current.strip():
        chunks.append(current.strip())
    return chunks


def main():
    embedder = SentenceTransformer(EMBED_MODEL)
    client = get_client()

    # rebuild the collection from scratch so ingestion is repeatable
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    collection = client.create_collection(COLLECTION_NAME)

    documents, ids, metadatas = [], [], []
    for path in sorted(glob.glob(os.path.join(DOCS_DIR, "*.txt"))):
        with open(path, encoding="utf-8") as f:
            text = f.read()
        source = os.path.basename(path)
        for i, chunk in enumerate(chunk_text(text)):
            documents.append(chunk)
            ids.append(f"{source}-{i}")
            metadatas.append({"source": source})

    embeddings = embedder.encode(documents).tolist()
    collection.add(documents=documents, embeddings=embeddings,
                   ids=ids, metadatas=metadatas)
    print(f"Ingested {len(documents)} chunks from {DOCS_DIR}/ "
          f"into collection '{COLLECTION_NAME}'")


if __name__ == "__main__":
    main()
'''

with open("ingest.py", "w") as f:
    f.write(ingest_script)

print("ingest.py written")

In [ ]:
# Cell 3 of 5: write the API server
# Same dual-mode client as ingest.py. All configuration comes from the
# environment, never from hardcoded values. This matters enormously for
# Docker, as Section 7 will make painfully clear.

rag_app = '''# app.py - RAG question-answering API
import os
from contextlib import asynccontextmanager
from typing import List

import chromadb
from fastapi import FastAPI, HTTPException
from openai import OpenAI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = os.getenv("COLLECTION_NAME", "demo_docs")
EMBED_MODEL = os.getenv("EMBED_MODEL", "all-MiniLM-L6-v2")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

state = {}


def get_client():
    host = os.getenv("CHROMA_HOST")
    if host:
        return chromadb.HttpClient(host=host, port=int(os.getenv("CHROMA_PORT", "8000")))
    return chromadb.PersistentClient(path=os.getenv("CHROMA_PATH", "chroma_db"))


@asynccontextmanager
async def lifespan(app: FastAPI):
    # load heavy resources once at startup
    state["embedder"] = SentenceTransformer(EMBED_MODEL)
    state["collection"] = get_client().get_or_create_collection(COLLECTION_NAME)
    state["llm"] = OpenAI()  # reads OPENAI_API_KEY from the environment
    yield
    state.clear()


app = FastAPI(title="RAG QA API", version="1.0.0", lifespan=lifespan)


class AskRequest(BaseModel):
    question: str
    top_k: int = 4


class AskResponse(BaseModel):
    answer: str
    sources: List[str]


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "collection": COLLECTION_NAME,
        "indexed_chunks": state["collection"].count(),
    }


@app.post("/ask", response_model=AskResponse)
def ask(req: AskRequest):
    if not req.question.strip():
        raise HTTPException(status_code=422, detail="Question is empty")

    # 1. embed the question
    query_vec = state["embedder"].encode(req.question).tolist()

    # 2. retrieve the most relevant chunks
    results = state["collection"].query(query_embeddings=[query_vec],
                                        n_results=req.top_k)
    chunks = results["documents"][0] if results["documents"] else []
    context = "\\n\\n".join(chunks)

    # 3. generate the answer
    completion = state["llm"].chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system",
             "content": "Answer using ONLY the provided context. "
                        "If the context is insufficient, say so."},
            {"role": "user",
             "content": f"Context:\\n{context}\\n\\nQuestion: {req.question}"},
        ],
    )
    return AskResponse(answer=completion.choices[0].message.content,
                       sources=chunks)
'''

with open("app.py", "w") as f:
    f.write(rag_app)

print("app.py written")

In [ ]:
# Cell 4 of 5: write requirements.txt
# Pin everything. "It worked last month" is not a deployment strategy.

requirements = """fastapi==0.115.6
uvicorn==0.34.0
chromadb==0.5.23
openai==1.59.7
sentence-transformers==3.3.1
pydantic==2.10.4
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt written")
print()
print("Heads up: sentence-transformers pulls in torch. That single line is")
print("responsible for most of the 2GB image we are about to build, and most")
print("of the savings in Section 5.")

In [ ]:
# Cell 5 of 5: write .dockerignore
# This is NOT optional for AI projects. Without it, Docker copies your venv,
# your local vector store, and any data folders into the build context.
# Builds get slow, and worse, secrets can leak into the image.

dockerignore = """.git
.gitignore
.env
venv/
.venv/
__pycache__/
*.pyc
chroma_db/
*.ipynb
README.md
"""

with open(".dockerignore", "w") as f:
    f.write(dockerignore)

print(".dockerignore written")
print()
print("Rule of thumb: if it is in .gitignore, it probably belongs in")
print(".dockerignore too. Especially .env. Section 7 shows exactly why.")
print()
print("Note that docs/ is NOT ignored: we want the knowledge base inside the")
print("image, because the build will ingest it. chroma_db/ IS ignored: any")
print("local index on your machine should not leak in; the image builds its own.")

### Quick Sanity Check (Optional, Outside Docker)

If you want to confirm the app works on bare metal first:

```bash
pip install -r requirements.txt
python ingest.py                 # builds chroma_db/ locally, embedded mode
export OPENAI_API_KEY=sk-...     # Windows PowerShell: $env:OPENAI_API_KEY="sk-..."
uvicorn app:app --port 8000
curl -X POST localhost:8000/ask -H "Content-Type: application/json" \
     -d '{"question": "What is a multi-stage build?"}'
```

And that is exactly the problem: those four manual steps, in the right order, with the right Python, on the right OS. Now we make all of it disappear into one image.

---
## 4. Dockerfile Basics and Best Practices for Python AI Apps

A Dockerfile is a recipe: each instruction creates a layer of the image. Here is the straightforward single-stage version for our app. Read the comments; every line earns its place.

One design decision to call out before you read it: the Dockerfile **runs `ingest.py` at build time** in embedded mode. That bakes the demo index into the image itself. The result is a fully self-contained image: anyone who pulls it can immediately ask questions, no setup, no external vector DB. For a small demo corpus this is the right call. For large or private corpora you would mount a volume or run a Chroma server instead, and we do exactly that in Section 6.

In [ ]:
# Write the single-stage Dockerfile

dockerfile_v1 = '''# Dockerfile - RAG QA API (single-stage version)

# 1. Base image: pin the version, use slim, never use latest.
#    slim = Debian without build tools and docs (~130MB vs ~900MB full)
#    Do NOT use alpine for AI apps: musl libc breaks torch/tokenizers wheels.
FROM python:3.11-slim

# 2. Working directory: every following command runs from /app
WORKDIR /app

# 3. Dependencies FIRST, code SECOND. This is the layer caching trick.
#    If requirements.txt is unchanged, Docker reuses the cached pip layer.
#    With torch in the mix, that cached layer saves you 3-5 minutes per rebuild.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 4. Pre-download the embedding model into the image.
#    Without this, every fresh container downloads ~90MB from HuggingFace
#    at startup. Bad for startup time, terrible for air-gapped deployments.
RUN python -c "from sentence_transformers import SentenceTransformer; \\
               SentenceTransformer('all-MiniLM-L6-v2')"

# 5. Copy the knowledge base and the code (these change often, so they go last)
COPY docs/ ./docs/
COPY ingest.py app.py ./

# 6. Build the vector index INSIDE the image (embedded mode: CHROMA_HOST is
#    not set, so ingest.py writes to /app/chroma_db). The image now ships
#    with its knowledge pre-loaded.
RUN python ingest.py

# 7. Run as a non-root user. If someone exploits your API, they should not
#    get root inside the container.
RUN useradd --create-home appuser && chown -R appuser /app
USER appuser

# 8. Document the port (this does not publish it; -p at run time does)
EXPOSE 8000

# 9. Default command, exec form (JSON array). Avoids shell quirks and makes
#    sure uvicorn receives stop signals properly.
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile_v1)

print("Dockerfile written")
print()
print(dockerfile_v1)

### Best Practices Checklist for Python AI Apps

The Dockerfile above already follows these. Keep this list next to you whenever you write one:

1. **Pin the base image.** `python:3.11-slim`, never `python:latest`. Your build should produce the same image next month.
2. **Slim, not alpine.** Alpine uses musl libc; many compiled ML wheels (torch, tokenizers, numpy in some versions) either fail to install or fall back to slow source builds.
3. **Dependencies before code.** `COPY requirements.txt` + `RUN pip install` before copying code. Code changes daily, dependencies change weekly. The cache split saves minutes per rebuild.
4. **`--no-cache-dir` on pip.** The pip download cache is dead weight inside an image. With torch involved that is several hundred MB.
5. **Use a `.dockerignore`.** Keep venvs, local vector stores, notebooks, and `.env` out of the build context.
6. **Decide where models AND data live, explicitly.** We bake the small embedding model and the demo index into the image: fast, reliable, self-contained. For multi-GB model weights or private corpora, mount volumes or use external services instead. The point is to make it a decision, not an accident.
7. **Non-root user.** One `useradd` line. Cheap insurance.
8. **One process per container.** The API is one container. A Chroma server is another. Do not cram both into one image; that is what Compose is for (Section 6).
9. **Exec-form CMD.** `CMD ["uvicorn", ...]` not `CMD uvicorn ...`. Signals (and therefore graceful shutdown) work correctly.

> **Mantra: dependencies first, code last, secrets never.**

### Build, Run, Test

In [ ]:
# Run these in your terminal, from the project folder:

print("""
# Build the image
$ docker build -t rag-app:1.0.0 .

[+] Building 214.8s (14/14) FINISHED
 => [1/9] FROM docker.io/library/python:3.11-slim                    2.1s
 => [2/9] WORKDIR /app                                               0.1s
 => [3/9] COPY requirements.txt .                                    0.0s
 => [4/9] RUN pip install --no-cache-dir -r requirements.txt       142.6s  <- torch...
 => [5/9] RUN python -c "from sentence_transformers import ..."     21.3s
 => [6/9] COPY docs/ ./docs/                                         0.0s
 => [7/9] COPY ingest.py app.py ./                                   0.0s
 => [8/9] RUN python ingest.py                                       24.7s  <- index baked in
 => [9/9] RUN useradd --create-home appuser && chown -R appuser      0.4s
 => exporting to image                                              19.1s
 => naming to docker.io/library/rag-app:1.0.0

# Run it. The API key is injected at runtime, never built in (Section 7).
$ docker run -d --name my-rag -p 8000:8000 \\
      -e OPENAI_API_KEY=sk-proj-... rag-app:1.0.0

# Health check: note indexed_chunks > 0, proof the baked index is there
$ curl localhost:8000/health
{"status":"healthy","collection":"demo_docs","indexed_chunks":7}

# Ask it something it learned from docs/
$ curl -X POST localhost:8000/ask \\
      -H "Content-Type: application/json" \\
      -d '{"question": "Why should I copy requirements.txt before my code?"}'

{"answer":"Because Docker caches layers. If requirements.txt has not changed,
the expensive pip install layer is reused, so a code-only change rebuilds in
seconds...","sources":["The most important caching rule. ..."]}

# Interactive docs in the browser: http://localhost:8000/docs
""")

> **Instructor's note:** Do the build live and narrate the layer steps as they scroll past. When the pip layer takes its two minutes, that is your moment to explain caching: rebuild immediately after with a one-character change to app.py and let the class watch steps 1-5 fly by as CACHED.

### Now Face the Truth About Size

In [ ]:
# Run in your terminal:

print("""
$ docker images | grep rag-app
rag-app    1.0.0    8f3a1c2d9e44    2 minutes ago    2.18GB

# 2.18 GB. For an API that is maybe 100 lines of Python plus three text files.
# Where did it all go? Ask docker history:

$ docker history rag-app:1.0.0
IMAGE          CREATED          CREATED BY                                SIZE
8f3a1c2d9e44   2 minutes ago    CMD ["uvicorn" "app:app" ...]             0B
...            2 minutes ago    RUN useradd --create-home appuser ...     4kB
...            2 minutes ago    RUN python ingest.py                      1.2MB  <- index
...            3 minutes ago    COPY ingest.py app.py ./                  7kB
...            3 minutes ago    COPY docs/ ./docs/                        4kB
...            3 minutes ago    RUN python -c "from sentence_tran...      91MB   <- model
...            5 minutes ago    RUN pip install --no-cache-dir ...        1.87GB <- THIS
...            6 minutes ago    WORKDIR /app                              0B
...            6 minutes ago    FROM python:3.11-slim                     130MB
""")

1.87GB of that image is pip packages, and the bulk of it is torch plus its CUDA-related dependencies, which we do not even use because this container runs CPU inference for a small embedding model. The baked index, by contrast, costs about 1MB. Knowledge is cheap; frameworks are heavy.

A 2GB image is not just an aesthetic problem:

- Pushing it to Docker Hub on classroom wifi takes a long time. Your peer pulling it takes just as long.
- Cloud deployments pull the image on every scale-up event. Big image = slow autoscaling.
- Registries charge for storage and bandwidth.

### Checkpoint 2

1. Why does `COPY requirements.txt` come before `COPY ingest.py app.py`?
2. What is the difference between `RUN` and `CMD`? Our Dockerfile uses `RUN python ingest.py`. What would change if it were `CMD python ingest.py`?
3. `EXPOSE 8000` is in the Dockerfile but the API is unreachable from your browser. What did you forget on the `docker run` command?
4. Your build context upload says "transferring context: 4.2GB". What is the most likely cause, and which file fixes it?

---
## 5. Multi-Stage Builds: From 2GB to ~500MB

### The Idea

A lot of what pip needs at **install time** (build tools, compilers, the pip cache machinery itself) is useless at **run time**. A multi-stage build uses two `FROM` statements:

- **Stage 1 (builder):** has everything needed to install and compile dependencies. Gets thrown away.
- **Stage 2 (runtime):** starts clean, and copies in only the finished virtual environment from the builder.

```
STAGE 1: builder                    STAGE 2: runtime
+---------------------------+       +---------------------------+
| python:3.11-slim          |       | python:3.11-slim          |
| + build-essential, gcc    |       |                           |
| + pip install everything  |  -->  | COPY /opt/venv (only the  |
|   into /opt/venv          | copy  |   installed packages)     |
| (this whole stage is      |       | + docs, ingest.py, app.py |
|  DISCARDED after build)   |       | = final image             |
+---------------------------+       +---------------------------+
```

There is a second, AI-specific trick stacked on top: install the **CPU-only torch wheel**. The default `pip install torch` on Linux pulls the CUDA build, which is enormous. Our embedding model runs fine on CPU.

In [ ]:
# Write the multi-stage Dockerfile

dockerfile_multistage = '''# Dockerfile.multistage - RAG QA API (multi-stage version)

# ============ STAGE 1: builder ============
FROM python:3.11-slim AS builder

# Build tools live here and ONLY here. They never reach the final image.
RUN apt-get update && apt-get install -y --no-install-recommends \\
    build-essential \\
    && rm -rf /var/lib/apt/lists/*

# Install everything into an isolated venv we can copy out later
RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

COPY requirements.txt .

# CPU-only torch first (the big win), then the rest of the requirements.
# The extra index tells pip to use the CPU wheel instead of the CUDA build.
RUN pip install --no-cache-dir torch==2.5.1 \\
        --index-url https://download.pytorch.org/whl/cpu \\
    && pip install --no-cache-dir -r requirements.txt

# Pre-download the embedding model in the builder; we copy the cache over
RUN python -c "from sentence_transformers import SentenceTransformer; \\
               SentenceTransformer('all-MiniLM-L6-v2')"

# ============ STAGE 2: runtime ============
FROM python:3.11-slim

WORKDIR /app

# Copy ONLY the finished venv and the model cache from the builder
COPY --from=builder /opt/venv /opt/venv
COPY --from=builder /root/.cache/huggingface /home/appuser/.cache/huggingface
ENV PATH="/opt/venv/bin:$PATH"
ENV HF_HOME=/home/appuser/.cache/huggingface

# Knowledge base + code, then bake the index, exactly as in the single-stage file
COPY docs/ ./docs/
COPY ingest.py app.py ./
RUN python ingest.py

RUN useradd --create-home appuser && chown -R appuser /app /home/appuser
USER appuser

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open("Dockerfile.multistage", "w") as f:
    f.write(dockerfile_multistage)

print("Dockerfile.multistage written")

In [ ]:
# Build both and compare. Run in your terminal:

print("""
$ docker build -t rag-app:1.0.0-slim -f Dockerfile.multistage .

[+] Building 118.2s (18/18) FINISHED
 ...
 => naming to docker.io/library/rag-app:1.0.0-slim

$ docker images | grep rag-app
rag-app    1.0.0          8f3a1c2d9e44    25 minutes ago    2.18GB
rag-app    1.0.0-slim     3b7e5f8a2c11    40 seconds ago    524MB

# Same app. Same baked index. Same behavior. Roughly a quarter of the size.

# Prove the behavior is identical:
$ docker run -d --name slim-rag -p 8002:8000 \\
      -e OPENAI_API_KEY=sk-proj-... rag-app:1.0.0-slim
$ curl localhost:8002/health
{"status":"healthy","collection":"demo_docs","indexed_chunks":7}
""")

### Where the Savings Came From

| Change | Approx. saving |
|---|---|
| CPU-only torch wheel instead of the default CUDA build | ~1.3GB |
| Build tools and apt caches left behind in the builder stage | ~250MB |
| No pip cache, no leftover wheels in the final image | ~80MB |
| **Final image** | **~2.18GB down to ~524MB** |

A few honest caveats, because students always ask:

- If your app genuinely needs GPU inference, you keep the CUDA torch build and start from an `nvidia/cuda` base image instead. The image will be large and that is fine; the multi-stage pattern still removes build tools.
- The exact numbers vary by version. The point is the technique, not the digits.
- Multi-stage builds also help with **secrets at build time**: anything used only in the builder stage never appears in the final image layers.

### Checkpoint 3

1. In a multi-stage build, what happens to everything in the builder stage that is not explicitly copied with `COPY --from=builder`?
2. Why does the default `pip install torch` make Linux images so large?
3. Your colleague says "just delete the build tools with a `RUN apt-get remove` at the end of a single-stage build". Why does that not actually shrink the image? (Hint: layers are append-only.)

---
## 6. Docker Compose: App + Vector DB + Monitoring

### One Container Was Never Going to Be Enough

The baked-in embedded index is perfect for a self-contained demo image. But a real deployment usually separates concerns: the API scales independently, the vector DB holds growing data, and something watches both. Our RAG system becomes three services:

```
                 +-------------------+
   user -------> |  rag-api          |
   POST /ask     |  (our FastAPI     |
                 |   container)      |
                 +---------+---------+
                           |
              query embeddings via HTTP
                           |
                 +---------v---------+        +-------------------+
                 |  chromadb         |        |  prometheus       |
                 |  (vector store,   |        |  (scrapes /metrics|
                 |   own container)  |        |   from rag-api)   |
                 +-------------------+        +-------------------+
```

You could start all three with three separate `docker run` commands, manually create a network, and remember the right flags every time. Nobody does that. **Docker Compose** describes the whole stack in one YAML file, and `docker compose up` starts everything.

And here is where the dual-mode `get_client()` we wrote in Section 3 pays off: setting one environment variable, `CHROMA_HOST`, flips the **same image** from embedded mode to server mode. No rebuild.

### The Compose File

In [ ]:
# Write docker-compose.yml

compose = '''# docker-compose.yml - RAG stack: API + vector DB + monitoring
services:

  rag-api:
    build:
      context: .
      dockerfile: Dockerfile.multistage
    image: rag-app:1.0.0-slim
    ports:
      - "8000:8000"             # host:container
    environment:
      - CHROMA_HOST=chromadb    # flips app.py/ingest.py into server mode;
                                # service name = hostname, see note below
      - CHROMA_PORT=8000
      - COLLECTION_NAME=demo_docs
    env_file:
      - .env                    # OPENAI_API_KEY lives here, NOT in this file
    depends_on:
      - chromadb
    restart: unless-stopped

  chromadb:
    image: chromadb/chroma:0.5.23
    ports:
      - "8001:8000"             # exposed on host 8001 so it does not clash with the API
    volumes:
      - chroma_data:/chroma/chroma    # persist the index across restarts
    restart: unless-stopped

  prometheus:
    image: prom/prometheus:v3.1.0
    ports:
      - "9090:9090"
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml:ro

volumes:
  chroma_data:
'''

with open("docker-compose.yml", "w") as f:
    f.write(compose)

print("docker-compose.yml written")

In [ ]:
# Write the Prometheus scrape config

prom_config = '''# prometheus.yml - scrape the RAG API metrics endpoint every 15s
global:
  scrape_interval: 15s

scrape_configs:
  - job_name: rag-api
    static_configs:
      - targets: ["rag-api:8000"]   # service name again, not localhost
'''

with open("prometheus.yml", "w") as f:
    f.write(prom_config)

print("prometheus.yml written")
print()
print("To expose /metrics from FastAPI, add one dependency and two lines:")
print()
print("  add to requirements.txt:  prometheus-fastapi-instrumentator==7.0.0")
print()
print("  in app.py, after creating the app:")
print("  from prometheus_fastapi_instrumentator import Instrumentator")
print("  Instrumentator().instrument(app).expose(app)")
print()
print("That gives you request counts, latencies, and error rates per endpoint,")
print("which is exactly what you want to watch on a RAG service in production.")

### The Single Most Important Compose Concept: Service Names Are Hostnames

Notice `CHROMA_HOST=chromadb` in the compose file. Compose puts every service on a shared private network and registers each service name in DNS. From inside the `rag-api` container, the vector DB is reachable at `http://chromadb:8000`.

This is exactly why `app.py` and `ingest.py` read `CHROMA_HOST` from the environment instead of hardcoding anything. The same image works:

- standalone (`CHROMA_HOST` unset, uses the index baked into the image)
- inside Compose (`CHROMA_HOST=chromadb`, talks to the Chroma container)
- in the cloud (`CHROMA_HOST=chroma.internal.mycompany.com`)

One image, three environments, zero code changes. This pattern has a name worth remembering: **configuration through the environment**.

### Driving the Stack

In [ ]:
# Run these in your terminal:

print("""
# Start everything (builds rag-api if needed)
$ docker compose up -d
[+] Running 4/4
 - Network rag-app_default        Created
 - Container rag-app-chromadb-1   Started
 - Container rag-app-prometheus-1 Started
 - Container rag-app-rag-api-1    Started

# See the stack
$ docker compose ps
NAME                    IMAGE                    STATUS         PORTS
rag-app-chromadb-1      chromadb/chroma:0.5.23   Up 10 seconds  0.0.0.0:8001->8000/tcp
rag-app-prometheus-1    prom/prometheus:v3.1.0   Up 10 seconds  0.0.0.0:9090->9090/tcp
rag-app-rag-api-1       rag-app:1.0.0-slim       Up 9 seconds   0.0.0.0:8000->8000/tcp

# One extra step in server mode: the Chroma CONTAINER starts empty, so run
# ingestion once against it. Same script, same image; CHROMA_HOST inside the
# container points it at the server instead of the baked local folder.
$ docker compose exec rag-api python ingest.py
Ingested 7 chunks from docs/ into collection 'demo_docs'

# Tail the API logs
$ docker compose logs -f rag-api

# Test end to end
$ curl -X POST http://localhost:8000/ask \\
      -H "Content-Type: application/json" \\
      -d '{"question": "What is layer caching in Docker?"}'

# Prometheus UI:  http://localhost:9090   (query: http_requests_total)
# API docs:       http://localhost:8000/docs

# Stop everything (keeps the chroma volume, so the index survives)
$ docker compose down

# Stop everything AND delete the vector index volume (fresh start)
$ docker compose down -v
""")

> **Instructor's note:** Do this live. Bring the stack up, run the ingest, fire two or three questions at `/ask`, then open Prometheus and graph `http_requests_total` so students see their own requests appear. It lands much better than a slide.

### Checkpoint 4

1. Why is `CHROMA_HOST` set to `chromadb` and not `localhost`? What is wrong with `localhost` inside a container?
2. What is the purpose of the `chroma_data` volume? What would happen on `docker compose down` without it?
3. `depends_on` makes chromadb start before rag-api. Does it guarantee chromadb is *ready* to accept connections? (Answer: no. Look up healthchecks if you hit this.)
4. Both rag-api and chromadb listen on port 8000 inside their containers. Why is there no conflict?
5. In Compose mode, why did we have to run `ingest.py` once, when the standalone image needed no ingestion at all?

---
## 7. Environment Variables and Secrets Management

### The Cardinal Rule

**Never bake a secret into an image.** Not in the Dockerfile, not in a copied `.env`, not in a build arg. Here is why this rule has no exceptions:

In [ ]:
# Why secrets in images are unrecoverable. Run in your terminal:

print("""
# Suppose someone wrote this in a Dockerfile:
#   ENV OPENAI_API_KEY=sk-proj-abc123...
# and pushed the image. Anyone who pulls it can do:

$ docker history --no-trunc bad-app:1.0.0
... /bin/sh -c #(nop) ENV OPENAI_API_KEY=sk-proj-abc123...   <- right there

# Or simply:
$ docker run --rm bad-app:1.0.0 env | grep OPENAI
OPENAI_API_KEY=sk-proj-abc123...

# Build args are NOT safer:
#   ARG OPENAI_API_KEY
# also shows up in docker history.

# Once pushed, consider the key compromised. Rotate it. There is no undo.
""")

The image is the **recipe and the kitchen**. The secrets are **handed to the cook at runtime**.

### The Right Way: Inject at Runtime

Three mechanisms, from quick to production-grade:

**1. The `-e` flag** (quick tests only; it ends up in your shell history)

```bash
docker run -e OPENAI_API_KEY=sk-... -p 8000:8000 rag-app:1.0.0-slim
```

**2. An `--env-file`** (the standard for local development)

```bash
# .env  (in .gitignore AND .dockerignore)
OPENAI_API_KEY=sk-proj-...
OPENAI_MODEL=gpt-4o-mini

docker run --env-file .env -p 8000:8000 rag-app:1.0.0-slim
```

Compose does the same thing with the `env_file:` key, which we already used in Section 6.

**3. A secrets manager** (production: AWS Secrets Manager, GCP Secret Manager, Vault, or Docker/Kubernetes secrets). The principle is identical: the orchestrator injects the value into the container at start time, and it never touches the image or the repo. We will not set one up today, but know that this is where the `--env-file` pattern graduates to.

### What CAN Live in the Dockerfile

`ENV` in a Dockerfile is fine for **non-secret defaults**: model names, port numbers, log levels. Our app already follows the split:

| Value | Secret? | Where it lives |
|---|---|---|
| `OPENAI_API_KEY` | Yes | `.env`, injected at runtime |
| `CHROMA_HOST` | No | unset for standalone mode; compose `environment:` for server mode |
| `OPENAI_MODEL` default | No | `os.getenv(..., "gpt-4o-mini")` default in code, overridable |
| `EMBED_MODEL`, `COLLECTION_NAME` | No | same pattern: sensible default in code, overridable from outside |

### The Env Var Contract

When you share an image (in 15 minutes you will), the environment variables ARE your public interface. Document them. A teammate pulling `yourusername/rag-app:1.0.0` needs to know, without reading your source:

```
Required:
  OPENAI_API_KEY     the runner supplies their OWN key
Optional:
  CHROMA_HOST        unset = embedded index baked into the image (default);
                     set it to point at an external Chroma server
  CHROMA_PORT        default: 8000 (only used when CHROMA_HOST is set)
  OPENAI_MODEL       default: gpt-4o-mini
  COLLECTION_NAME    default: demo_docs
Ports:
  8000               HTTP API
```

That block belongs in your repo README and in the Docker Hub repository description.

### Checkpoint 5

1. A key was committed in a Dockerfile and the image was pushed publicly for 10 minutes before being deleted. What is the correct response?
2. Why is `.env` in both `.gitignore` and `.dockerignore`?
3. Your peer runs your image and gets `openai.AuthenticationError`. Is your image broken?

---
## 8. Packaging and Sharing RAG Applications

### Tag Like You Mean It

By default Docker tags images `latest`. In any shared or production setting, `latest` is an anti-pattern: you cannot tell what is running and you cannot roll back. Use explicit versions:

```bash
rag-app:1.0.0          # semantic version: Major.Minor.Patch
rag-app:1.1.0          # added reranking
rag-app:1.1.1          # fixed the chunk overlap bug
rag-app:a3b4c5d        # git hash, used by CI/CD pipelines
```

Two rules that will save you at 2am someday:

1. Every deployment is reproducible: given a tag, you can recreate exactly that container.
2. Rollback is just deploying the previous tag.

### Push to Docker Hub

In [ ]:
# Replace with your own Docker Hub username before the exercise
DOCKER_HUB_USERNAME = "yourusername"   # <- change this
IMAGE_NAME = "rag-app"
VERSION = "1.0.0"

full_tag = f"{DOCKER_HUB_USERNAME}/{IMAGE_NAME}:{VERSION}"

print(f"""
# Step 1: Log in
$ docker login
Username: {DOCKER_HUB_USERNAME}
Password: ************
Login Succeeded

# Step 2: Tag the multi-stage image with your namespace
$ docker tag rag-app:1.0.0-slim {full_tag}

# Step 3: Push. This is where the Section 5 work pays off:
# pushing ~520MB instead of 2.18GB on classroom wifi.
$ docker push {full_tag}
The push refers to repository [docker.io/{DOCKER_HUB_USERNAME}/{IMAGE_NAME}]
5f70bf18a086: Pushed
...
{VERSION}: digest: sha256:9c4e... size: 2415
""")

### No Registry? `docker save` and `docker load`

Sometimes there is no registry available: an air-gapped environment, a client demo, or simply hub.docker.com being slow. You can move images as plain files:

```bash
# On your machine: export the image to a tarball
docker save rag-app:1.0.0-slim -o rag-app-1.0.0.tar

# Hand it over on a USB stick, shared drive, whatever

# On their machine: import it
docker load -i rag-app-1.0.0.tar
docker run --env-file .env -p 8000:8000 rag-app:1.0.0-slim
```

### The Shipping Checklist for a RAG Image

When you hand someone a RAG application as an image, the image alone is not the whole deliverable. Ship this set:

1. **The image** on a registry, with an explicit version tag.
2. **The env var contract** (Section 7): required and optional variables, defaults, ports.
3. **A one-line run command** they can copy-paste, plus a sample request:
   ```bash
   docker run --env-file .env -p 8000:8000 yourusername/rag-app:1.0.0
   curl -X POST localhost:8000/ask -H "Content-Type: application/json" \
        -d '{"question": "What is a multi-stage build?"}'
   ```
4. **The compose file**, if the app is meant to run with an external vector DB and monitoring.
5. **A note about data.** Our demo image bakes a small public corpus in, so it works out of the box. State that explicitly. If your real corpus is large or private, document the alternative: a volume mount for a pre-built index, or an ingestion step against a Chroma server. Never bake a customer's documents into a public image.

> **Mental model:** the image is the contract between you and whoever runs your app. If they need to read your source code to run it, the contract is incomplete.

---
# Live Exercise: Ship Your RAG App to a Peer

This is the moment of truth for everything above. You will personalize the RAG app we built today, containerize it, push it, and a classmate will run it on their machine. The personalization step is how your peer proves they are really running **your** image and not just rebuilding the shared demo.

**Total time: ~25-30 minutes. Work in pairs assigned now.**

> **Instructor's note:** Pair students before starting and write the pairs on the board. Keep a known-good fallback image on the course Docker Hub account so anyone whose build fails can still complete Part C as the "runner". Walk the room during Part A; the missing env-file and the forgotten rebuild after editing docs are where people get stuck.

### Part A: Personalize and Containerize (~15 min)

You already have every file from the cells above. Now make the image yours:

1. **Add a personal document to the knowledge base.** Create `docs/04_about_the_builder.txt` with three or four sentences only you would write. For example:

   ```
   About the builder of this image.

   This image was built by <your name> during the AI Engineering Docker
   class. The builder's favorite debugging command is docker logs, and
   their hometown is <city>. The secret word for this image is <pick one>.
   ```

   This is the verification payload: your peer will ask the API for the secret word.
2. **Rebuild with the new document.** Watch the cache do its job: the giant pip layer is reused, only the docs/ingest layers rerun.
   ```bash
   docker build -t rag-app:1.0.0 -f Dockerfile.multistage .
   ```
3. **Create your `.env`** (if you have not already) with your own `OPENAI_API_KEY`. Confirm `.env` is listed in `.dockerignore`.
4. **Run and test locally:**
   ```bash
   docker run -d --name my-rag --env-file .env -p 8000:8000 rag-app:1.0.0
   curl localhost:8000/health
   curl -X POST localhost:8000/ask -H "Content-Type: application/json" \
        -d '{"question": "What is the secret word for this image?"}'
   docker logs my-rag        # if anything fails, the answer is in here
   ```
   If the answer contains your secret word, your index is baked in correctly.

### Part B: Push (~5 min)

```bash
docker login
docker tag rag-app:1.0.0 YOUR_USERNAME/rag-app:1.0.0
docker push YOUR_USERNAME/rag-app:1.0.0
```

While it pushes, send your peer two things in chat: your Docker Hub username, and your env var contract (one required variable, `OPENAI_API_KEY`; port 8000; everything else has defaults).

### Part C: Run Your Peer's Image (~10 min)

Now switch hats. You are the operations engineer receiving someone else's application.

```bash
# Pull their image
docker pull PEER_USERNAME/rag-app:1.0.0

# Run it with YOUR OWN .env file. You supply your own OPENAI_API_KEY.
# Never share API keys, not even with your lab partner.
docker run -d --name peer-rag --env-file .env -p 8080:8000 \
    PEER_USERNAME/rag-app:1.0.0

# Test it
curl localhost:8080/health

# The verification question:
curl -X POST localhost:8080/ask -H "Content-Type: application/json" \
     -d '{"question": "Who built this image, and what is the secret word?"}'
```

If the answer names your peer and their secret word, you have just run someone else's complete AI application, embedding model, vector index and all, with a single pull and a single run command. Notice everything you did NOT do: no pip install, no Python version check, no sqlite patching, no ingestion. That is the entire point of today.

### Success Criteria

You are done when ALL of these are true:

- [ ] Your peer pulled your image with zero conversation about Python versions, pip, or sqlite
- [ ] Their `curl /health` against your image shows `indexed_chunks` greater than zero
- [ ] Their verification question returns your name and secret word
- [ ] Your `.env` never left your machine

### Troubleshooting Table

| Symptom | Likely cause | Fix |
|---|---|---|
| `port is already allocated` | Your own container still holds 8000 | Map a different host port: `-p 8080:8000` |
| `openai.AuthenticationError` | Env file not passed, or wrong var name | Check `--env-file .env` and that the variable is named `OPENAI_API_KEY` |
| `/health` shows `indexed_chunks: 0` | Edited docs/ but ran the OLD image, or `RUN python ingest.py` missing from the Dockerfile | Rebuild, confirm the ingest step appears in the build log |
| Secret word missing from the answer | Same as above, or the question is too vague | Rebuild and ask directly: "What is the secret word?" |
| Push/pull painfully slow | 2GB single-stage image | Use the multi-stage build (Section 5), tag and push again |
| Container exits immediately | Crash at startup | `docker logs <name>` always tells you why |
| `connection refused` on curl | Container still loading the embedding model | Wait 5-10 seconds, check `docker logs` for "Application startup complete" |

> **Instructor's note:** When pairs finish early, hand them the design question: "your real corpus is 50GB of private PDFs; do you still bake the index into the image?" Surface the answers for the whole class. The options worth drawing out are (a) ship an ingestion script or endpoint, (b) mount a volume with a pre-built index, (c) run a shared Chroma server as in Section 6. Baked-in only makes sense for small public demo corpora like ours.

---
## Practice Exercises

Complete these before the next class:

### Exercise 1 - Layer Caching, Measured
1. Build the image and note the total build time.
2. Add a sentence to one file in `docs/` and rebuild. Note the time and which layers say CACHED.
3. Add one package to `requirements.txt` and rebuild. Note the time.
4. Write two sentences explaining the difference.

### Exercise 2 - Shrink It Further
1. Run `docker history` on both your images and identify the three largest layers in each.
2. Try a smaller embedding model (for example `paraphrase-MiniLM-L3-v2`) and rebuild. Measure the difference.
3. Target: under 600MB. Post your final size in the course channel.

### Exercise 3 - Complete the Compose Stack
1. Add `prometheus-fastapi-instrumentator==7.0.0` to requirements, wire it into `app.py` as shown in Section 6, and rebuild.
2. Bring up the full compose stack, run the ingest step, and confirm metrics appear at http://localhost:9090.
3. Optional stretch: add a `grafana/grafana` service and build a dashboard with request latency for `/ask`.

### Exercise 4 - The Offline Handoff
1. `docker save` your image to a tarball and note its size.
2. Exchange tarballs with a different classmate (not your in-class partner) via a shared drive.
3. `docker load` and run theirs. Confirm the env var contract was enough to run it without messaging them.

### Exercise 5 - Make It Your Own
Swap the `docs/` folder for a corpus you actually care about (your own notes, a project README, a few blog posts), rebuild, and push it as `YOUR_USERNAME/rag-app:1.1.0`. Congratulations, you now maintain a versioned, shippable AI product.

---
## What's Next

### You Can Now Ship Any Python AI Application

After today you can:

- Explain precisely why "works on my machine" happens, and prevent it
- Build a complete RAG service (ingestion, vector store, API) and wrap it in a production-shaped Dockerfile with caching, a non-root user, and a proper `.dockerignore`
- Cut image size by 4x with multi-stage builds and CPU-only wheels
- Describe a multi-service RAG stack (API + vector DB + monitoring) in one compose file
- Flip the same image between embedded and server mode with a single environment variable
- Handle secrets the right way: injected at runtime, never baked in
- Package and hand off a RAG application that a stranger can run from a single command

The image you pushed today is the exact artifact a cloud platform would deploy. Nothing about it is "classroom only".

### Next Class: Deploying the Containerized RAG App to the Cloud

We take `yourusername/rag-app:1.0.0` from Docker Hub and put it behind a public HTTPS endpoint on a managed container service, with the vector DB and monitoring alongside it.

**Before next class:**
1. Your image is pushed to Docker Hub (today's exercise)
2. You have a cloud account set up (instructions in the course channel)
3. Exercises 1 and 2 above are done; image size matters for cloud pulls

### The Bigger Arc

```
Today                     Next class               After that
----------------------    ---------------------    ----------------------
Build and containerize    Deploy the container     CI/CD: rebuild and
the RAG app           --> to the cloud behind  --> redeploy automatically
(Docker, Compose)         a public endpoint        on every git push
```

The same application, moving step by step through a real production pipeline.

---

### Reference Materials

| Resource | Link |
|---|---|
| Docker Official Docs | https://docs.docker.com/ |
| Dockerfile Best Practices | https://docs.docker.com/build/building/best-practices/ |
| Multi-Stage Builds | https://docs.docker.com/build/building/multi-stage/ |
| Docker Compose Docs | https://docs.docker.com/compose/ |
| ChromaDB Docs (client modes, deployment) | https://docs.trychroma.com/ |
| PyTorch CPU Wheels | https://pytorch.org/get-started/locally/ |
| FastAPI in Containers | https://fastapi.tiangolo.com/deployment/docker/ |
| prometheus-fastapi-instrumentator | https://github.com/trallnag/prometheus-fastapi-instrumentator |

Docker beginner crash course video: [YouTube](https://www.youtube.com/watch?v=pg19Z8LL06w)